In [100]:
def parse_transcript(transcipt_name):

    chat_stats = {}
    policy = None
    dialogs = {}
    DIALOG_TYPES = ['cts', 'cts_llm']

    user_dialog_nums = {}

    with open(transcipt_name, "r") as transcript:
        for line in transcript:
            if "(POLICY:" in line:
                current_dialog = {}
                tmp = line.split()
                user = tmp[1].strip()
                policy = tmp[3].strip(")").strip()
                current_dialog['user'] = user
                current_dialog['turns'] = []
                # TODO: add back goals to dialogs and analyze if there is anything interesting at a per/goal level
                # goal_type = tmp[-1].strip(")").strip()
            elif "USER:" in line and not "POST-NLU" in line:
                current_dialog["turns"].append(line)
            elif "SYSTEM" in line:
                current_dialog['turns'].append(line)
            elif "DIALOG END:" in line:
                current_dialog["end_condition"] = line.split(":")[1].strip()
            elif "SUBJECTIVE LENGTH" in line:
                current_dialog["sub_length"] = line.split(":")[1].strip()
            elif "SUBJECTIVE QUALITY" in line:
                current_dialog["sub_quality"] = line.split(":")[1].strip()
            elif line.strip() == "":
                # TODO: Change this line to analyse one group at a time
                if current_dialog and policy in DIALOG_TYPES:
                    obj_length = len(current_dialog["turns"])
                    if not policy in chat_stats:
                        chat_stats[policy] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}
                    chat_stats[policy]["length"].append(obj_length)
                    chat_stats[policy]["end_condition"].append(current_dialog["end_condition"])
                    if "sub_length" in current_dialog:
                        chat_stats[policy]["sub_length"].append(int(current_dialog["sub_length"]))
                    if "sub_quality" in current_dialog:
                        chat_stats[policy]["sub_quality"].append(int(current_dialog["sub_quality"]))
                    current_dialog["length"] = obj_length
                    if current_dialog['user'] not in user_dialog_nums:
                        user_dialog_nums[current_dialog['user']] = 0
                    user_dialog_nums[current_dialog['user']] += 1
                    if policy not in dialogs:
                        dialogs[policy] = []
                    dialogs[policy].append(current_dialog)
                    current_dialog = {}
    return (chat_stats, dialogs, user_dialog_nums)

In [101]:
chat_stats, dialogs, user_dialog_nums = parse_transcript("main_study/clean_transcript.txt")
baseline_chat_stats, baseline_dialogs, baseline_user_dialog_nums = parse_transcript("../mental_models/combined_data/transcript.txt")

In [102]:
def calc_dialog_metrics(chat_stats, dialogs):
    num_dialogs = {}
    success = {}
    avg_len_dialog = {}
    avg_len_first_utterance = {}
    avg_len_all_utterances = {}
    avg_sub_quality = {}
    avg_sub_len = {}

    for policy in chat_stats:
        num_dialogs[policy] = 0
        success[policy] = 0
        avg_len_dialog[policy] = 0
        avg_len_first_utterance[policy] = 0
        avg_len_all_utterances[policy] = 0
        avg_sub_quality[policy] = 0
        avg_sub_len[policy] = 0
        
        success[policy] = chat_stats[policy]["end_condition"].count("SUCCESS") + chat_stats[policy]["end_condition"].count("SUCCESS - OTHER QUESTION")
        avg_len_dialog[policy] = sum(chat_stats[policy]["length"])
        avg_sub_quality[policy] = sum(chat_stats[policy]["sub_quality"])
        avg_sub_len[policy] = sum(chat_stats[policy]["sub_length"])

        for record in dialogs[policy]:
            num_dialogs[policy] += 1
            d = record['turns']
            user_turns = [t for t in d if "USER" in t]
            len_user_turns = [len(t[6:].split()) for t in user_turns]
            avg_len_first_utterance[policy] += len_user_turns[0]
            avg_len_all_utterances[policy] += sum(len_user_turns)/len(len_user_turns)

        avg_len_dialog[policy] = avg_len_dialog[policy]/num_dialogs[policy]
        avg_sub_len[policy] = avg_sub_len[policy]/num_dialogs[policy]
        avg_sub_quality[policy] = avg_sub_quality[policy]/num_dialogs[policy]

        avg_len_first_utterance[policy] = avg_len_first_utterance[policy]/num_dialogs[policy]
        avg_len_all_utterances[policy] = avg_len_all_utterances[policy]/num_dialogs[policy]

        print(policy)
        print(f"NUM DIALOGS: {num_dialogs[policy]}")
        print(f"COUNT SUCCESS: {success[policy]}")
        print(f"PERCENT SUCCESS: {success[policy]/num_dialogs[policy]*100}")
        print(f"SUBJECTIVE QUALITY: {avg_sub_quality[policy]}")
        print(f"AVG NUM TURNS: {avg_len_dialog[policy]}")
        print(f"SUBJECTIVE LENGTH: {avg_sub_len[policy]}")

        print(f"AVG LEN INITIAL UTTERANCE: {avg_len_first_utterance[policy]}")
        print(f"AVG LEN ALL UTTERANCES: {avg_len_all_utterances[policy]}")

In [103]:
print('BASELINE STATS')
calc_dialog_metrics(baseline_chat_stats, baseline_dialogs)
print("\n")
print('LLM CTS STATS')

calc_dialog_metrics(chat_stats, dialogs)


BASELINE STATS
cts
NUM DIALOGS: 61
COUNT SUCCESS: 47
PERCENT SUCCESS: 77.04918032786885
SUBJECTIVE QUALITY: 2.8688524590163933
AVG NUM TURNS: 7.377049180327869
SUBJECTIVE LENGTH: 2.918032786885246
AVG LEN INITIAL UTTERANCE: 8.721311475409836
AVG LEN ALL UTTERANCES: 6.361865729898516


LLM CTS STATS
cts_llm
NUM DIALOGS: 66
COUNT SUCCESS: 59
PERCENT SUCCESS: 89.39393939393939
SUBJECTIVE QUALITY: 2.9545454545454546
AVG NUM TURNS: 10.090909090909092
SUBJECTIVE LENGTH: 2.757575757575758
AVG LEN INITIAL UTTERANCE: 10.363636363636363
AVG LEN ALL UTTERANCES: 6.7469926538108345


## Statistic for Study

**Facit**: Significant

- Better success
- Better perceived answer satisfaction/quality
- Longer dialogs, but subjective length was not perceived longer

In [108]:
from scipy.stats import barnard_exact
yeses = [47, 60]
nos = [14, 8]
print(barnard_exact([yeses, nos], alternative="less"))

BarnardExactResult(statistic=-1.6865150947530607, pvalue=0.0491901918028183)


In [111]:
import ast
import csv

def parse_surveys(survey_file, outfile, user_dialog_nums):
    post_surveys = []
    pre_surveys = []
    users = set()
    with open(survey_file, "r") as infile:
        for line in infile:
            if "PREFERRED_STYLE" not in line:
                user, survey = line.split("||")
                user = user.split(":")[1].strip()
                users.add(user)
                survey = survey[13:].strip()
                survey = ast.literal_eval(survey)
                survey["user"] = user
                if "POST-SURVEY" in line:
                    post_surveys.append(survey)
                else:
                    pre_surveys.append(survey)   

    # Remove Users who don't interact with the system
    user_black_list = set()
    user_gray_list = set()

    for user in users:
        if user in user_dialog_nums:
            if user_dialog_nums[user] != 3:
                user_gray_list.add(user)
        else:
            user_black_list.add(user)

    print(f"Removed {len(user_black_list)} users: {user_black_list}")
    print("To Investigate: ", user_gray_list)
    users = [user for user in users if user not in user_black_list]
    print(len(users))

    if not outfile is None:
        # Save surveys to CSV for easier conent analysis
        with open("pre_survey.csv", "w", newline='') as outfile:
            fieldnames = pre_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in pre_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

        with open("post_survey.csv", "w", newline='') as outfile:
            fieldnames = post_surveys[0].keys()
            writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter="|")
            writer.writeheader()
            for s in post_surveys:
                if s['user'] in user_black_list:
                    continue
                writer.writerow(s)

    return pre_surveys, post_surveys, user_black_list

In [112]:
baseline_pre_surveys, baseline_post_surveys, base_blacklist = parse_surveys("../mental_models/combined_data/survey_log_full.txt", None, baseline_user_dialog_nums)
exp_pre_surveys, exp_post_surveys, exp_blacklist = parse_surveys("main_study/combined_survey.txt", None, user_dialog_nums=user_dialog_nums)

Removed 45 users: {'5bbe9193fb7d45e2e10d3aade3aa89', '597bc392e979365272ccf5960b6679', '266f4afc0f4957543ecd710668a801', 'bf4b5adecf96a4d3eccdcf42e32d91', '443650e58eea05ea703d12d8b7cf06', '0f97158978beebaeb59f4dda0db17c', 'eb7cdc7219ceeb7b2862154329eca8', 'ba401ddeae9aef6345d8c3dbddb3bf', '1122e007efd6d6c220883470890125', '3531297f0bd32d54dcc23e8e8a50ad', '09b1a7da8812a2efd58c0a1b92cc0b', '5a465be12adc3998b4fe940817cddc', 'e004be8181957be418d758d41b56e2', 'bf83f3984f247796d4437afe55ffa1', '85452be9505ef3ee3499d59b5300f3', '232cd8e82603855aebc8caa4d53fc1', 'e100425dee079289c5660a0b2db5d1', '63fe6663ab1740ce3d095bd3ec270f', 'ed81efaafec6d941adbdf96636bd6d', '47c97ed8e9ba2195e48f663a9f35a7', '767d6d0026f6568ed1fbe324f434ee', '92c512341aec50139fc63dab2be508', '1d6610aff8f95079a27b87ed0804c5', '80dce4aa2bb6e4747dea90c91ca9fe', '7f946e73fd33c3a2f15beac9248fc6', '6e6ed1b576c8f271b519a059d5bf0d', 'd3f5d2249e51c387c882cb587ba9ca', 'c951b710a16924c54f90bccc29460b', '3d9670b634a7a64e7a91bbd34c18

In [59]:
import numpy as np

def parse_trust_and_usability_scores(post_surveys, user_black_list):
    trust = []
    reliability = []
    usability = []

    u_usability = {}
    u_trust = {}
    u_reliability = {}

    for res in post_surveys:
        user = res["user"]
        if user in user_black_list or user == "47e68725c26f72d805709141e76fd0" or user == "57affbcf1a53cf8152a4f84b337572":
            continue
        user_reliability = (int(res["reliability_1"]) + int(res["reliability_2"]) + (6 - int(res["reliability_3"])) + int(res["reliability_4"]) + (6 - int(res["reliability_5"])) + int(res["reliability_6"])) / 6
        user_trust = (int(res["trust_1"]) + int(res["trust_2"])) / 2
        # should be 0 to 4 scale, not 1 to 5
        user_usability = ((int(res['umux_1']) - 1) + (5 - int(res['umux_2'])) + (int(res['umux_3']) - 1) + (5 - int(res['umux_4']))) / 16 * 100
        u_usability[user] = user_usability
        trust.append(user_trust)
        u_trust[user] = user_trust
        reliability.append(user_reliability)
        u_reliability[user] = user_reliability
        usability.append(user_usability)
        
        # print(f" USER: {user}: Trust: {user_trust} Reliability: {user_reliability} Usability: {user_usability}")
        
    print(f"TRUST: {np.mean(trust)} +/- {np.std(trust)}")
    print(f"RELIABILITY: {np.mean(reliability)} +/- {np.std(reliability)}")
    print(f"USABILITY: {np.mean(usability)} +/- {np.std(usability)}")
    return trust, reliability, usability

In [62]:
from scipy.stats import ttest_ind
print('BASELINE')
baseline_trust, baseline_reliability, baseline_usability = parse_trust_and_usability_scores(baseline_post_surveys, base_blacklist)
print("\nLLM CTS")
exp_trust, exp_reliability, exp_usability = parse_trust_and_usability_scores(exp_post_surveys, exp_blacklist)

print("\n")
print("trust")
print(ttest_ind(baseline_trust, exp_trust))

print("reliability")
print(ttest_ind(baseline_reliability, exp_reliability))

print("usability")
print(ttest_ind(baseline_usability, exp_usability))

BASELINE
TRUST: 3.1578947368421053 +/- 0.8119604537127112
RELIABILITY: 2.9649122807017547 +/- 0.7083027802872404
USABILITY: 62.828947368421055 +/- 23.073260416776552

LLM CTS
TRUST: 3.0 +/- 1.2339883600452934
RELIABILITY: 3.0606060606060606 +/- 0.9892674753276016
USABILITY: 67.89772727272727 +/- 25.02097518843271


trust
Ttest_indResult(statistic=0.4640802253827979, pvalue=0.6451703942024913)
reliability
Ttest_indResult(statistic=-0.3423688584899265, pvalue=0.7339116511488598)
usability
Ttest_indResult(statistic=-0.653943322711666, pvalue=0.5169860056849688)


### Test Simulation Results for Significance

In [1]:
from scipy.stats import barnard_exact

def calc_simulation_significance(exp_success: float):
    cts_reimbuse = 73.86
    yeses = [5 * cts_reimbuse, 5 * exp_success]
    nos = [5 * (100 - cts_reimbuse), 5 * (100 - exp_success)]

    print(barnard_exact([yeses, nos]))

In [3]:
llm_reimuburse = 77

calc_simulation_significance(exp_success=llm_reimuburse)

BarnardExactResult(statistic=-1.1211111237359948, pvalue=0.2899568563215845)


## Get Demographic Info

In [114]:
import csv
user_stats_per_policy = {}
user_exp_map = {}
with open('pre_survey.csv', 'r') as infile:
    reader = csv.DictReader(infile, delimiter="|")
    for row in reader:
        keep_keys = ['gender', 'age', 'experience_chatbots', 'experience_businesstravel', 'user']
        policy = 'cts_llm'
        if policy not in user_stats_per_policy:
            user_stats_per_policy[policy] = []
        exp = int(row['experience_chatbots'])
        user = row['user']
        user_exp_map[user] = exp
        user_stats_per_policy[policy].append({key: row[key] for key in keep_keys})

In [115]:
# get gender information
gender_distribution = {}
for policy in user_stats_per_policy:
    gender_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        gender = entry['gender']
        if gender not in gender_distribution[policy]:
            gender_distribution[policy][gender] = 0
        gender_distribution[policy][gender] += 1

print(gender_distribution)

{'cts_llm': {'female': 13, 'male': 11}}


In [116]:
# get age information
age_distribution = {}
for policy in user_stats_per_policy:
    age_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        age = entry['age']
        if age not in age_distribution[policy]:
            age_distribution[policy][age] = 0
        age_distribution[policy][age] += 1

print(age_distribution)

{'cts_llm': {'40-49': 3, '20-29': 15, '30-39': 3, '50-59': 1, '<20': 2}}


In [117]:
# get previous chatbot experience information
chatbot_exp_distribution = {}
chatbot_avg_exp = []
for policy in user_stats_per_policy:
    chatbot_exp_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        chatbot_exp = int(entry['experience_chatbots'])
        chatbot_avg_exp.append(chatbot_exp)
        if chatbot_exp not in chatbot_exp_distribution[policy]:
            chatbot_exp_distribution[policy][chatbot_exp] = 0
        chatbot_exp_distribution[policy][chatbot_exp] += 1

print(chatbot_exp_distribution)
print(sum(chatbot_avg_exp)/len(chatbot_avg_exp))

{'cts_llm': {3: 11, 2: 3, 4: 5, 5: 5}}
3.5


In [118]:
# get previous business travel experience information
bt_exp_distribution = {}
bt_avg_exp = []
for policy in user_stats_per_policy:
    bt_exp_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        bt_exp = entry['experience_businesstravel']
        bt_avg_exp.append(int(bt_exp))
        if bt_exp not in bt_exp_distribution[policy]:
            bt_exp_distribution[policy][bt_exp] = 0
        bt_exp_distribution[policy][bt_exp] += 1

print(bt_exp_distribution)
print(sum(bt_avg_exp)/len(bt_avg_exp))

{'cts_llm': {'3': 10, '1': 8, '2': 2, '4': 2, '5': 2}}
2.5
